# Curriculum Design Analysis: Zero-Shot Baseline Review

**Objective:** Analyze zero-shot performance across categories to design an informed curriculum training strategy.

**Approach:**
1. Load zero-shot evaluation results from all models
2. Identify weak categories (low accuracy across models)
3. Visualize performance gaps
4. Map weak areas to curriculum stages
5. Create presentation-ready figures

**Expected Output:** Slides showing curriculum justification based on data-driven analysis

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

BASE_PATH = Path('/home/g2/temporal-news-reasoning')
RESULTS_PATH = BASE_PATH / 'results' / 'baseline' / 'zero_shot'
TRAINING_PATH = BASE_PATH / 'data' / 'training'

print("Paths initialized successfully")

In [ ]:
## Section 1: Load Zero-Shot Evaluation Results

# Function to find and load benchmark reports
def load_reports_by_benchmark(benchmark_name):
    """Load all reports for a given benchmark (time, tram, timebench)"""
    reports = {}
    
    # Try direct reports
    for report_file in RESULTS_PATH.glob(f"*_{benchmark_name}_report.json"):
        model_name = report_file.name.split('_')[0]
        with open(report_file) as f:
            reports[model_name] = json.load(f)
    
    # Try nested directories (mistral, qwen3.5)
    for model_dir in RESULTS_PATH.iterdir():
        if model_dir.is_dir() and model_dir.name not in ['recovery_backups', 'results']:
            report_path = model_dir / benchmark_name / f"{model_dir.name.lower()}_{benchmark_name}_report.json"
            if not report_path.exists():
                # Try alternative naming
                report_files = list((model_dir / benchmark_name).glob("*_report.json"))
                if report_files:
                    report_path = report_files[0]
            
            if report_path.exists():
                model_name = model_dir.name.lower()
                with open(report_path) as f:
                    reports[model_name] = json.load(f)
    
    return reports

# Load reports for both benchmarks
time_reports = load_reports_by_benchmark("time")
tram_reports = load_reports_by_benchmark("tram")

print(f"TIME benchmark - Models loaded: {list(time_reports.keys())}")
print(f"TRAM benchmark - Models loaded: {list(tram_reports.keys())}")

In [ ]:
## Display loaded reports structure
print("=" * 60)
print("TIME BENCHMARK - Category Performance by Model")
print("=" * 60)
for model_name, report in time_reports.items():
    print(f"\n{model_name.upper()} - Overall Accuracy: {report['overall_accuracy']:.2%}")
    for cat, metrics in report['by_category'].items():
        print(f"  {cat:25s}: {metrics['accuracy']:.2%} ({metrics['correct']}/{metrics['total']})")

print("\n" + "=" * 60)
print("TRAM BENCHMARK - Category Performance by Model")
print("=" * 60)
for model_name, report in tram_reports.items():
    print(f"\n{model_name.upper()} - Overall Accuracy: {report['overall_accuracy']:.2%}")
    for cat, metrics in report['by_category'].items():
        print(f"  {cat:25s}: {metrics['accuracy']:.2%} ({metrics['correct']}/{metrics['total']})")

In [ ]:
## Section 2: Identify Weak Categories (Failure Modes Analysis)

def get_category_matrix(reports):
    """Convert reports to a DataFrame for easy comparison"""
    data = []
    for model_name, report in reports.items():
        for category, metrics in report['by_category'].items():
            data.append({
                'model': model_name,
                'category': category,
                'accuracy': metrics['accuracy'],
                'correct': metrics['correct'],
                'total': metrics['total']
            })
    return pd.DataFrame(data)

time_df = get_category_matrix(time_reports)
tram_df = get_category_matrix(tram_reports)

# Compute weak categories (average accuracy < threshold)
print("\n" + "=" * 60)
print("WEAK CATEGORIES ANALYSIS: Average Zero-Shot Performance")
print("=" * 60)

# TIME benchmark
print("\nTIME Benchmark:")
time_avg = time_df.groupby('category')['accuracy'].agg(['mean', 'std', 'min', 'max'])
time_avg = time_avg.sort_values('mean')
print(time_avg.to_string())

# TRAM benchmark
print("\n\nTRAM Benchmark:")
tram_avg = tram_df.groupby('category')['accuracy'].agg(['mean', 'std', 'min', 'max'])
tram_avg = tram_avg.sort_values('mean')
print(tram_avg.to_string())

# Combined weak categories (< 40% average accuracy)
weak_threshold = 0.40
print("\n" + "=" * 60)
print(f"CRITICAL WEAK AREAS (avg accuracy < {weak_threshold:.0%}):")
print("=" * 60)

weak_time = time_avg[time_avg['mean'] < weak_threshold]
weak_tram = tram_avg[tram_avg['mean'] < weak_threshold]

print(f"\nTIME: {list(weak_time.index)}")
print(f"TRAM: {list(weak_tram.index)}")

all_weak = set(weak_time.index) | set(weak_tram.index)
print(f"\nTarget categories for curriculum (both benchmarks): {sorted(all_weak)}")

In [ ]:
## Section 3: Create Visualizations for Slides

# Figure 1: Heatmap of model performance across categories (TIME)
fig, ax = plt.subplots(figsize=(14, 6))
time_pivot = time_df.pivot(index='category', columns='model', values='accuracy')
sns.heatmap(time_pivot, annot=True, fmt='.2%', cmap='RdYlGn', vmin=0, vmax=1, 
            cbar_kws={'label': 'Accuracy'}, ax=ax)
ax.set_title('Zero-Shot Performance Across Categories (TIME Benchmark)', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
plt.tight_layout()
plt.savefig(BASE_PATH / 'results' / 'curriculum_analysis_time_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Figure 2: Heatmap of model performance across categories (TRAM)
fig, ax = plt.subplots(figsize=(14, 10))
tram_pivot = tram_df.pivot(index='category', columns='model', values='accuracy')
sns.heatmap(tram_pivot, annot=True, fmt='.2%', cmap='RdYlGn', vmin=0, vmax=1, 
            cbar_kws={'label': 'Accuracy'}, ax=ax)
ax.set_title('Zero-Shot Performance Across Categories (TRAM Benchmark)', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Category', fontsize=12)
plt.tight_layout()
plt.savefig(BASE_PATH / 'results' / 'curriculum_analysis_tram_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Heatmaps saved")

In [ ]:
# Figure 3: Average performance by category (sorted - shows weak areas)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# TIME benchmark
time_avg_sorted = time_avg.sort_values('mean')
colors_time = ['red' if x < 0.4 else 'orange' if x < 0.6 else 'green' for x in time_avg_sorted['mean']]
axes[0].barh(range(len(time_avg_sorted)), time_avg_sorted['mean'], color=colors_time, alpha=0.7)
axes[0].set_yticks(range(len(time_avg_sorted)))
axes[0].set_yticklabels(time_avg_sorted.index)
axes[0].set_xlabel('Average Accuracy', fontsize=11)
axes[0].set_title('Average Zero-Shot Performance by Category\n(TIME Benchmark)', fontsize=12, fontweight='bold')
axes[0].axvline(x=0.4, color='red', linestyle='--', label='Weak threshold (40%)', alpha=0.5)
axes[0].axvline(x=0.6, color='orange', linestyle='--', label='Medium threshold (60%)', alpha=0.5)
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# TRAM benchmark
tram_avg_sorted = tram_avg.sort_values('mean')
colors_tram = ['red' if x < 0.4 else 'orange' if x < 0.6 else 'green' for x in tram_avg_sorted['mean']]
axes[1].barh(range(len(tram_avg_sorted)), tram_avg_sorted['mean'], color=colors_tram, alpha=0.7)
axes[1].set_yticks(range(len(tram_avg_sorted)))
axes[1].set_yticklabels(tram_avg_sorted.index)
axes[1].set_xlabel('Average Accuracy', fontsize=11)
axes[1].set_title('Average Zero-Shot Performance by Category\n(TRAM Benchmark)', fontsize=12, fontweight='bold')
axes[1].axvline(x=0.4, color='red', linestyle='--', label='Weak threshold (40%)', alpha=0.5)
axes[1].axvline(x=0.6, color='orange', linestyle='--', label='Medium threshold (60%)', alpha=0.5)
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_PATH / 'results' / 'curriculum_weak_areas.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Weak areas visualization saved")

In [ ]:
## Section 4: Analyze Training Data Distribution

# Sample training data to see category representation
def analyze_training_data():
    """Analyze category distribution in curriculum stages"""
    stages = {}
    
    for stage_file in sorted(TRAINING_PATH.glob('stage*.jsonl')):
        examples = []
        with open(stage_file) as f:
            for line in f:
                if line.strip():
                    examples.append(json.loads(line))
        
        stages[stage_file.stem] = examples
        print(f"\n{stage_file.stem}: {len(examples)} examples")
        
        # Try to extract category info from examples
        categories_found = set()
        for ex in examples[:5]:  # Show first 5 examples
            if 'category' in ex:
                categories_found.add(ex['category'])
            elif 'categories' in ex:
                categories_found.update(ex['categories'])
        
        if categories_found:
            print(f"  Sample categories: {categories_found}")
    
    return stages

training_stages = analyze_training_data()

print("\n" + "=" * 60)
print("TRAINING DATA SUMMARY")
print("=" * 60)
for stage_name, examples in training_stages.items():
    print(f"{stage_name}: {len(examples)} examples")

In [ ]:
## Section 5: Map Findings to Curriculum Strategy

# Define curriculum strategy based on weak categories
curriculum_strategy = {
    'Stage 1: Explicit Reasoning': {
        'focus': 'Strong foundation on clear temporal relationships',
        'target_categories': ['causality', 'temporal_ordering', 'temporal_simultaneous'],
        'rationale': 'Models show moderate performance; build core reasoning first'
    },
    'Stage 2: Implicit Reasoning': {
        'focus': 'Complex temporal inference without explicit markers',
        'target_categories': ['duration', 'arithmetic', 'temporal_context'],
        'rationale': 'Weak areas (12-44% accuracy); significant gap here'
    },
    'Stage 3: Complex Synthesis': {
        'focus': 'Combining multiple reasoning types under uncertainty',
        'target_categories': ['frequency', 'temporal_nli', 'temporal_relation'],
        'rationale': 'Reinforces weak reasoning on larger problem space'
    }
}

print("=" * 70)
print("CURRICULUM STRATEGY JUSTIFICATION")
print("=" * 70)

for stage, strategy in curriculum_strategy.items():
    print(f"\n{stage}")
    print(f"  Focus: {strategy['focus']}")
    print(f"  Target Categories: {', '.join(strategy['target_categories'])}")
    print(f"  Rationale: {strategy['rationale']}")

# Create summary table
print("\n" + "=" * 70)
print("WEAK CATEGORIES DISTRIBUTION ACROSS CURRICULUM")
print("=" * 70)

weak_categories_summary = pd.DataFrame({
    'Weak Category': sorted(all_weak),
    'TIME Avg Accuracy': [time_avg.loc[cat, 'mean'] if cat in time_avg.index else np.nan for cat in sorted(all_weak)],
    'TIME Min': [time_avg.loc[cat, 'min'] if cat in time_avg.index else np.nan for cat in sorted(all_weak)],
    'TRAM Avg Accuracy': [tram_avg.loc[cat, 'mean'] if cat in tram_avg.index else np.nan for cat in sorted(all_weak)],
    'TRAM Min': [tram_avg.loc[cat, 'min'] if cat in tram_avg.index else np.nan for cat in sorted(all_weak)],
})

print(weak_categories_summary.to_string(index=False))

In [ ]:
## Section 6: Create Comprehensive Summary Visualization

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Plot 1: Weak categories comparison
ax1 = fig.add_subplot(gs[0, :])
weak_comparison = weak_categories_summary[['Weak Category', 'TIME Avg Accuracy', 'TRAM Avg Accuracy']].set_index('Weak Category')
x = np.arange(len(weak_comparison))
width = 0.35
ax1.bar(x - width/2, weak_comparison['TIME Avg Accuracy'], width, label='TIME', alpha=0.8)
ax1.bar(x + width/2, weak_comparison['TRAM Avg Accuracy'], width, label='TRAM', alpha=0.8)
ax1.set_ylabel('Average Accuracy', fontsize=11)
ax1.set_title('Weak Categories Across Benchmarks (Zero-Shot Baseline)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(weak_comparison.index, rotation=45, ha='right')
ax1.legend()
ax1.axhline(y=0.4, color='red', linestyle='--', alpha=0.3, label='40% threshold')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Model comparison (average across all categories)
ax2 = fig.add_subplot(gs[1, 0])
model_avg_time = time_df.groupby('model')['accuracy'].mean().sort_values()
colors2 = plt.cm.viridis(np.linspace(0, 1, len(model_avg_time)))
ax2.barh(range(len(model_avg_time)), model_avg_time.values, color=colors2, alpha=0.7)
ax2.set_yticks(range(len(model_avg_time)))
ax2.set_yticklabels(model_avg_time.index)
ax2.set_xlabel('Average Accuracy', fontsize=10)
ax2.set_title('Model Performance - TIME Benchmark', fontsize=11, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Plot 3: Model comparison (TRAM)
ax3 = fig.add_subplot(gs[1, 1])
model_avg_tram = tram_df.groupby('model')['accuracy'].mean().sort_values()
ax3.barh(range(len(model_avg_tram)), model_avg_tram.values, color=colors2, alpha=0.7)
ax3.set_yticks(range(len(model_avg_tram)))
ax3.set_yticklabels(model_avg_tram.index)
ax3.set_xlabel('Average Accuracy', fontsize=10)
ax3.set_title('Model Performance - TRAM Benchmark', fontsize=11, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

# Plot 4: Curriculum stage strategy
ax4 = fig.add_subplot(gs[2, :])
ax4.axis('off')
strategy_text = """
CURRICULUM TRAINING STRATEGY (Justification from Zero-Shot Analysis):

Stage 1: EXPLICIT REASONING (1,187 examples)
  → Focus: Clear temporal relationships & causality inference
  → Because: Models show ~45-80% on these categories (buildable foundation)

Stage 2: IMPLICIT REASONING (700 examples)  
  → Focus: Duration, arithmetic, temporal context extraction
  → Because: Critical weak areas (12-44% zero-shot) requiring targeted training

Stage 3: COMPLEX SYNTHESIS (1,000 examples)
  → Focus: Frequency, temporal NLI, multi-relation reasoning
  → Because: Reinforce weak reasoning under complex scenarios
  
Expected Impact: Curriculum targets where ALL models fail (40% threshold), 
addressing fundamental reasoning gaps in systematic progression.
"""
ax4.text(0.05, 0.95, strategy_text, transform=ax4.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.savefig(BASE_PATH / 'results' / 'curriculum_strategy_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comprehensive summary visualization saved")

In [ ]:
## Section 7: Export Data for Presentation Use

# Save summary tables as CSV
weak_categories_summary.to_csv(BASE_PATH / 'results' / 'weak_categories_summary.csv', index=False)
time_avg.to_csv(BASE_PATH / 'results' / 'category_performance_time.csv')
tram_avg.to_csv(BASE_PATH / 'results' / 'category_performance_tram.csv')

# Save pivots for easy reference
time_pivot.to_csv(BASE_PATH / 'results' / 'model_category_matrix_time.csv')
tram_pivot.to_csv(BASE_PATH / 'results' / 'model_category_matrix_tram.csv')

print("✓ Exported CSV files:")
print("  - weak_categories_summary.csv")
print("  - category_performance_time.csv")
print("  - category_performance_tram.csv")
print("  - model_category_matrix_time.csv")
print("  - model_category_matrix_tram.csv")

# Create JSON report for slides
curriculum_report = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'models_analyzed': list(time_reports.keys()),
    'benchmarks': ['TIME', 'TRAM'],
    'weak_categories': sorted(list(all_weak)),
    'weak_threshold': 0.40,
    'curriculum_strategy': curriculum_strategy,
    'key_findings': {
        'most_challenging': {
            'TIME': time_avg_sorted.index[0],
            'TRAM': tram_avg_sorted.index[0]
        },
        'total_examples_in_curriculum': sum(len(examples) for examples in training_stages.values()),
        'models_to_train': ['mistral', 'llama', 'qwen3.5']
    }
}

with open(BASE_PATH / 'results' / 'curriculum_analysis_report.json', 'w') as f:
    json.dump(curriculum_report, f, indent=2)

print("\n✓ Exported JSON report: curriculum_analysis_report.json")

print("\n" + "=" * 70)
print("SUMMARY OF OUTPUTS FOR PRESENTATION")
print("=" * 70)
print("\nVisualizations (PNG):")
print("  1. curriculum_analysis_time_heatmap.png - Model vs Category performance (TIME)")
print("  2. curriculum_analysis_tram_heatmap.png - Model vs Category performance (TRAM)")
print("  3. curriculum_weak_areas.png - Sorted weak areas by both benchmarks")
print("  4. curriculum_strategy_summary.png - Full-page summary (ready for slides)")
print("\nData Files (CSV/JSON):")
print("  - weak_categories_summary.csv - Key weak categories")
print("  - curriculum_analysis_report.json - Complete analysis report")
print("\nRECOMMENDATION: Use curriculum_strategy_summary.png as main slide output")